# Retail — Spark ingestion

Generates a synthetic retail data set with Spark Connect and writes it to Iceberg through the
platform catalog. Runs as `retail.analyst`; the Spark session uses this person's catalog grants.

Tables:
* `retailref.reference.stores` — 200 stores
* `retailref.reference.products` — 5,000 products
* `retail.bronze.orders` — 1.5M synthetic orders, partitioned by `order_month`


In [ ]:
rows = 1500000


In [ ]:
import stardelt
from pyspark.sql import functions as F

spark = stardelt.spark()
spark.sql("CREATE SCHEMA IF NOT EXISTS retail.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS retailref.reference")
spark.sql("CREATE SCHEMA IF NOT EXISTS retail.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS retail.gold")
print("spark", spark.version)


In [ ]:
cities = ["Berlin","Hamburg","Munich","Cologne","Frankfurt","Leipzig","Dresden","Stuttgart","Dortmund","Bremen"]
countries = ["DE"] * len(cities)
stores = (spark.range(1, 201)
  .selectExpr("cast(id as int) as store_id",
              f"element_at(array({','.join(repr(c) for c in cities)}), cast(id % {len(cities)} + 1 as int)) as city",
              "cast('DE' as string) as country",
              "element_at(array('North','South','East','West'), cast(id % 4 + 1 as int)) as region",
              "concat('Store ', id) as store_name")
)
stores.writeTo("retailref.reference.stores").createOrReplace()
print("stores", stores.count())


In [ ]:
categories = ["Electronics","Home","Grocery","Fashion","Sports","Toys","Beauty","Books","Auto","Garden"]
products = (spark.range(1, 5001)
  .selectExpr("cast(id as int) as product_id",
              "concat('Product ', id) as product_name",
              f"element_at(array({','.join(repr(c) for c in categories)}), cast(id % {len(categories)} + 1 as int)) as category",
              "round(2 + (id * 7 % 20000) / 100.0, 2) as list_price",
              "round(1 + (id * 5 % 15000) / 100.0, 2) as cost")
)
products.writeTo("retailref.reference.products").createOrReplace()
print("products", products.count())


In [ ]:
orders = (spark.range(0, rows, 1, 8)
  .selectExpr(
    "id as order_id",
    "timestampadd(SECOND, cast((id % 365) * 86400 + (id * 37) % 86400 as int), timestamp'2023-01-01 00:00:00') as order_ts",
    "cast(id % 250000 + 1 as bigint) as customer_id",
    "cast(id % 5000 + 1 as int) as product_id",
    "cast(id % 200 + 1 as int) as store_id",
    "cast(id % 5 + 1 as int) as quantity",
    "round(5 + (id * 37 % 49500) / 100.0, 2) as unit_price",
    "round((id * 13 % 30) / 100.0, 2) as discount",
    "case cast(id % 4 as int) when 0 then 'web' when 1 then 'mobile' when 2 then 'store' else 'partner' end as channel",
    "case when id % 100 < 88 then 'completed' when id % 100 < 95 then 'cancelled' else 'returned' end as status"
  )
  .withColumn("order_date", F.to_date("order_ts"))
  .withColumn("order_month", F.date_format("order_ts", "yyyyMM").cast("int"))
)
orders.writeTo("retail.bronze.orders").using("iceberg").partitionedBy("order_month").createOrReplace()
print("orders", spark.table("retail.bronze.orders").count())


## Verify from the same Spark session

In [ ]:
spark.sql("SELECT count(*) AS rows, min(order_ts) AS first_order, max(order_ts) AS last_order, round(sum(quantity * unit_price * (1 - discount)), 2) AS gross FROM retail.bronze.orders").show(truncate=False)
spark.sql("SELECT status, count(*) AS n FROM retail.bronze.orders GROUP BY status ORDER BY n DESC").show()
